In [2]:
import requests, zipfile, io
import glob   #glob sert à trouver les fichiers dans ton dossier.
import pandas as pd
import xml.etree.ElementTree as ET    #ElementTree sert à lire et extraire les données contenues dans les fichiers xml.
from datetime import datetime

log_file = "log_file.txt"   #fichier journal des opérations ETL.
target_file = "transformed_data.csv"   #fichier contenant les données transformées.

#Ce script permet de télécharger un fichier zippé et de le dézipper
url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-PY0221EN-SkillsNetwork/labs/module%206/"
    "Lab%20-%20Extract%20Transform%20Load/data/source.zip"
)
r = requests.get(url)
with zipfile.ZipFile(io.BytesIO(r.content)) as zip_ref:
    zip_ref.extractall("data")

In [4]:
def extract_from_csv(file_to_process):   #Extait tous les fichiers csv
    df = pd.read_csv(file_to_process)
    return df
    
def extract_from_json(file_to_process):        #Extrait tous les fichiers json
    df = pd.read_json(file_to_process, lines = True)
    return df  
    
'''Ce script crée un dataframe vide avec des noms de colonnes, 
lie et extrait les données contenues dans les fichiers xml et insére les données extraites dans le dataframe vide'''
def extract_from_xml(file_to_process):       
    df = pd.DataFrame(columns=["name", "height", "weight"]) 
    tree = ET.parse(file_to_process) 
    root = tree.getroot() 
    for person in root: 
        name = person.find("name").text 
        height = float(person.find("height").text) 
        weight = float(person.find("weight").text) 
        df = pd.concat([df, pd.DataFrame([{"name":name, "height":height, "weight":weight}])], ignore_index = True) 
    return df 
    
#Ce script permet d'assembler tous les données extraites et les inséré dans un dataframe unique
def extract(): 

#Crée un dataframe vide pour tous les données extraites
    extracted_data = pd.DataFrame(columns=['name','height','weight']) 
        
    # traiter tous les fichiers csv sauf le fichier cible
    for csvfile in glob.glob("data/*.csv"): 
        if csvfile != target_file:      #Vérifier si le fichier ne correspond pas au fichier cible
            extracted_data = pd.concat([extracted_data, pd.DataFrame(extract_from_csv(csvfile))], ignore_index = True)
         
    # traiter tous les fichiers json 
    for jsonfile in glob.glob("data/*.json"): 
        extracted_data = pd.concat([extracted_data, pd.DataFrame(extract_from_json(jsonfile))], ignore_index = True) 
     
    # traiter tous les fichiers xml 
    for xmlfile in glob.glob("data/*.xml"): 
        extracted_data = pd.concat([extracted_data, pd.DataFrame(extract_from_xml(xmlfile))], ignore_index = True) 
         
    return extracted_data 

#Ce script transforme les données extraites
def transform(data): 
    '''Convert inches to meters and round off to two decimals 
    1 inch is 0.0254 meters '''
    data['height'] = round(data.height * 0.0254,2)    #Convesion d'unité
 
    '''Convert pounds to kilograms and round off to two decimals 
    1 pound is 0.45359237 kilograms '''
    data['weight'] = round(data.weight * 0.45359237,2)  #Conversion d'unité
	
    return data 

#Ce script charge les données transformées dans le fichier cible
def load_data(target_file, transformed_data): 
    transformed_data.to_csv(target_file, index = False)
    


#Ce script permet d'écrire des messages sur le fichier log et d'horodater
def log_progress(message): 
    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second 
    now = datetime.now() # get current timestamp 
    timestamp = now.strftime(timestamp_format) 
    with open(log_file,"a") as f: 
        f.write(timestamp + ',' + message + '\n') 


#Dans ce script on a les différentes messages à écrire dans le fichier log pour chaque étape du processus ETL
#Début du processus ETL
log_progress("ETL Job Started") 
 
#Début de l'extraction
log_progress("Extract phase Started") 
extracted_data = extract() 
 
#Fin de l'extraction
log_progress("Extract phase Ended") 
 
#Début de la transformation
log_progress("Transform phase Started") 
transformed_data = transform(extracted_data) 
print("Transformed Data") 
print(transformed_data) 
 
#Fin de la transformation
log_progress("Transform phase Ended") 

#Début du chargement
log_progress("Load phase Started") 
load_data(target_file,transformed_data) 
 
#Fin du chargement
log_progress("Load phase Ended") 
 
#Fin du processus ETL
log_progress("ETL Job Ended") 

C:\Users\baaro\AppData\Local\Temp\ipykernel_10436\1942731808.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  extracted_data = pd.concat([extracted_data, pd.DataFrame(extract_from_csv(csvfile))], ignore_index = True)
C:\Users\baaro\AppData\Local\Temp\ipykernel_10436\1942731808.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{"name":name, "height":height, "weight":weight}])], ignore_index = True)
C:\Users\baaro\AppData\Local\Temp\ipykernel_10436\19427318

Transformed Data
     name  height  weight
0    alex    1.67   51.25
1    ajay    1.82   61.91
2   alice    1.76   69.41
3    ravi    1.73   64.56
4     joe    1.72   65.45
5    alex    1.67   51.25
6    ajay    1.82   61.91
7   alice    1.76   69.41
8    ravi    1.73   64.56
9     joe    1.72   65.45
10   alex    1.67   51.25
11   ajay    1.82   61.91
12  alice    1.76   69.41
13   ravi    1.73   64.56
14    joe    1.72   65.45
15   jack    1.74   55.93
16    tom    1.77   64.18
17  tracy    1.78   61.90
18   john    1.72   50.97
19   jack    1.74   55.93
20    tom    1.77   64.18
21  tracy    1.78   61.90
22   john    1.72   50.97
23   jack    1.74   55.93
24    tom    1.77   64.18
25  tracy    1.78   61.90
26   john    1.72   50.97
27  simon    1.72   50.97
28  jacob    1.70   54.73
29  cindy    1.69   57.81
30   ivan    1.72   51.77
31  simon    1.72   50.97
32  jacob    1.70   54.73
33  cindy    1.69   57.81
34   ivan    1.72   51.77
35  simon    1.72   50.97
36  jacob    1.70   5